# 00 — Environment and reproducibility

The numerical analysis is organized so that each stage of the study can be reproduced independently, beginning from the classical Orbital Edge Computing model and introducing the quantum-optimization components only after the corresponding constrained formulation has been validated. This first notebook does not contain scientific results; it records the software environment, fixes the common random seed and verifies that the local package can be imported before the subsequent experiments are executed.

The separation is intentional. In particular, changes in the Qiskit stack or in a classical solver should not silently alter the physical OEC model, and the environment information reported here will therefore accompany every set of numerical results used in the paper.

In [ ]:
from __future__ import annotations

import importlib.metadata as metadata
import json
import platform
import sys
from pathlib import Path

import numpy as np

SEED = 20260929
rng = np.random.default_rng(SEED)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

print(f"Repository root: {ROOT}")
print(f"Common random seed: {SEED}")

## Software environment

The implementation uses an open-source classical reference solver for the first validation stages and keeps the quantum dependencies separate from the physical model. Package versions are read directly from the active Python environment, so that a saved notebook provides an explicit record of the software stack used to obtain its outputs.

In [ ]:
packages = [
    "numpy",
    "scipy",
    "pandas",
    "networkx",
    "matplotlib",
    "qiskit",
    "qiskit-optimization",
]

environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "seed": SEED,
    "packages": {},
}

for package in packages:
    try:
        environment["packages"][package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        environment["packages"][package] = None

environment

The quantum packages are allowed to be absent while the classical reference model is being developed. They become mandatory only when the QUBO and QAOA notebooks are reached; at that stage the tested versions will be frozen together with the corresponding simulation results.

In [ ]:
from oec_qaoa import canonical_c1

instance = canonical_c1()
print(
    f"Imported oec_qaoa successfully: {instance.name} contains "
    f"{len(instance.services)} services and {len(instance.edges)} graph edges."
)

## Reproducibility record

Once the numerical pipeline is frozen, this dictionary will also be serialized with the experiment outputs. Keeping the environment record separate from the scientific results makes it possible to distinguish a genuine change in the model from a difference introduced by a software update.

In [ ]:
output_dir = ROOT / "results" / "environment"
output_dir.mkdir(parents=True, exist_ok=True)

with (output_dir / "environment.json").open("w", encoding="utf-8") as file:
    json.dump(environment, file, indent=2)

print(output_dir / "environment.json")